# Task 5: Sales Prediction using Python

**Author**: Pratham Bhat  
**Objective**: Predict sales using Linear Regression based on advertising spend  
**Dataset**: Advertising spend data (user-provided CSV)  
**Model**: Linear Regression (TV, Radio, Newspaper → Sales)

## 1. Load Libraries and Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

In [ ]:
# COLUMN NAME CONFIG - Edit if your CSV has different column names
TV_COLUMN = 'TV'
RADIO_COLUMN = 'Radio'
NEWSPAPER_COLUMN = 'Newspaper'
SALES_COLUMN = 'Sales'

# Load data
df = pd.read_csv('advertising.csv')

# Remove unnamed index column if present
if 'Unnamed: 0' in df.columns:
    df = df.drop('Unnamed: 0', axis=1)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nStatistics:")
print(df.describe())

## 2. Exploratory Data Analysis

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
correlation = df.corr()
sns.heatmap(correlation, annot=True, fmt='.3f', cmap='coolwarm', center=0, ax=ax, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
ax.set_title('Advertising Dataset - Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Channel Analysis - Scatter plots with regression lines
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Sales vs Advertising Channels', fontsize=14, fontweight='bold')

channels = [(TV_COLUMN, 'TV', 'steelblue'),
            (RADIO_COLUMN, 'Radio', 'coral'),
            (NEWSPAPER_COLUMN, 'Newspaper', 'mediumseagreen')]

for idx, (col, label, color) in enumerate(channels):
    ax = axes[idx]
    ax.scatter(df[col], df[SALES_COLUMN], alpha=0.6, s=60, color=color, edgecolors='black', linewidth=0.5)
    
    # Regression line
    z = np.polyfit(df[col], df[SALES_COLUMN], 1)
    p = np.poly1d(z)
    ax.plot(df[col], p(df[col]), "r--", linewidth=2, label='Linear fit')
    
    # Correlation
    corr = df[col].corr(df[SALES_COLUMN])
    ax.set_xlabel(f'{label} Spend ($1000s)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Sales ($1000s)', fontsize=11, fontweight='bold')
    ax.set_title(f'{label} (Correlation: {corr:.3f})', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Distribution Analysis
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Distribution Analysis', fontsize=14, fontweight='bold')

axes[0, 0].hist(df[TV_COLUMN], bins=30, color='steelblue', alpha=0.7, edgecolor='black')
axes[0, 0].set_xlabel('TV Spend ($1000s)', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0, 0].set_title('TV Spend Distribution', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3, axis='y')

axes[0, 1].hist(df[RADIO_COLUMN], bins=30, color='coral', alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('Radio Spend ($1000s)', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Radio Spend Distribution', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='y')

axes[1, 0].hist(df[NEWSPAPER_COLUMN], bins=30, color='mediumseagreen', alpha=0.7, edgecolor='black')
axes[1, 0].set_xlabel('Newspaper Spend ($1000s)', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Newspaper Spend Distribution', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

axes[1, 1].hist(df[SALES_COLUMN], bins=30, color='purple', alpha=0.7, edgecolor='black')
axes[1, 1].set_xlabel('Sales ($1000s)', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Sales Distribution', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 3. Model Training

In [ ]:
# Prepare data
X = df[[TV_COLUMN, RADIO_COLUMN, NEWSPAPER_COLUMN]]
y = df[SALES_COLUMN]

# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X, y)

# Predictions
y_pred = lr_model.predict(X)

# Metrics
mae = mean_absolute_error(y, y_pred)
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print(f"Model Coefficients:")
print(f"  Intercept: {lr_model.intercept_:.6f}")
print(f"  TV: {lr_model.coef_[0]:.6f}")
print(f"  Radio: {lr_model.coef_[1]:.6f}")
print(f"  Newspaper: {lr_model.coef_[2]:.6f}")
print(f"\nModel Performance:")
print(f"  R² Score: {r2:.4f}")
print(f"  MAE: {mae:.4f}")
print(f"  RMSE: {rmse:.4f}")

## 4. Model Evaluation

In [ ]:
# Actual vs Predicted
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(y, y_pred, alpha=0.6, s=60, color='steelblue', edgecolors='navy', linewidth=0.5)
ax.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=2, label='Perfect Prediction')
ax.set_xlabel('Actual Sales ($1000s)', fontsize=12, fontweight='bold')
ax.set_ylabel('Predicted Sales ($1000s)', fontsize=12, fontweight='bold')
ax.set_title(f'Actual vs Predicted Sales (R² = {r2:.4f}, MAE = {mae:.4f})', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Residuals Analysis
residuals = y - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Predicted
axes[0].scatter(y_pred, residuals, alpha=0.6, s=60, color='steelblue', edgecolors='navy', linewidth=0.5)
axes[0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0].set_xlabel('Predicted Sales ($1000s)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Residuals ($1000s)', fontsize=11, fontweight='bold')
axes[0].set_title('Residuals vs Predicted Values', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Residuals distribution
axes[1].hist(residuals, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Residuals ($1000s)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[1].set_title(f'Residuals Distribution (Mean: {residuals.mean():.4f})', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Feature Coefficients (Impact on Sales)
fig, ax = plt.subplots(figsize=(10, 6))
channels_names = ['TV', 'Radio', 'Newspaper']
coefficients = lr_model.coef_
colors = ['steelblue', 'coral', 'mediumseagreen']

bars = ax.barh(channels_names, coefficients, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
ax.set_xlabel('Coefficient Value', fontsize=12, fontweight='bold')
ax.set_title('Feature Coefficients - Impact on Sales', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Add value labels
for bar, coef in zip(bars, coefficients):
    width = bar.get_width()
    ax.text(width, bar.get_y() + bar.get_height()/2.,
            f'{coef:.4f}', ha='left' if coef > 0 else 'right', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Summary & Interpretation

**Model Equation:**
```
Sales = {lr_model.intercept_:.4f} + {lr_model.coef_[0]:.4f}×TV + {lr_model.coef_[1]:.4f}×Radio + {lr_model.coef_[2]:.4f}×Newspaper
```

**Coefficient Interpretation:**
- Higher coefficient = stronger impact on sales
- TV coefficient indicates sales increase per $1000 spent on TV
- R² shows % of variance explained by the model

**Performance:**
- R² = {r2:.4f} (explains {r2*100:.2f}% of sales variance)
- MAE = {mae:.4f} (average prediction error in $1000s)